# ShopFlow — Phase 3: ANALYTICS Star Schema
### Transform STAGED → ANALYTICS · star schema · fact + dimensions · business queries

---

### What this notebook does

Builds 5 tables in `SHOPFLOW_ANALYTICS` — the layer analysts and dashboards actually query:

| Table | Grain | Source |
|-------|-------|--------|
| `FACT_ORDERS` | One row per order item | `STG_ORDER_ITEMS` JOIN `STG_ORDERS` |
| `DIM_CUSTOMERS` | One row per unique customer | `STG_CUSTOMERS` LEFT JOIN `STG_GEOLOCATION` |
| `DIM_PRODUCTS` | One row per product | `STG_PRODUCTS` |
| `DIM_SELLERS` | One row per seller | `STG_SELLERS` LEFT JOIN `STG_GEOLOCATION` |
| `DIM_DATE` | One row per calendar day | Generated — no source table |

Then runs the 4 core business queries the entire project was built for.

### Why a star schema?

A star schema puts one fact table at the centre surrounded by dimension tables. Every analytical question is answered by joining the fact to one or more dimensions:

```text
              DIM_CUSTOMERS
                    │
DIM_DATE ── FACT_ORDERS ── DIM_PRODUCTS
                    │
              DIM_SELLERS
```

Analysts write simple JOINs. BI tools generate SQL automatically. The schema is self-documenting — anyone can look at it and understand the business model.

### Pattern used — same CTAS as Phase 2

Every ANALYTICS table is built with:
```sql
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.table AS
SELECT ... FROM SHOPFLOW_STAGED.STG_* ...;
```
STAGED is the only source — ANALYTICS never reads from RAW directly.

### STAGED stays untouched

ANALYTICS reads from STAGED but never modifies it. If you need to fix a dimension, drop and recreate only that table.

---

### Run cells one at a time — top to bottom

Each cell = one statement = one visible result.

---

### Key Snowflake concepts in this notebook

| Topic | Cells |
|-------|-------|
| Star schema design — fact vs dimension | All table cells |
| `DISTINCT` in dimension builds | DIM_CUSTOMERS, DIM_SELLERS |
| `DATE_TRUNC` and `DATEADD` | DIM_DATE generator |
| `GENERATOR` + `FLATTEN` — row generation | DIM_DATE |
| Clustering keys — `CLUSTER BY` | FACT_ORDERS clustering cell |
| `SYSTEM$CLUSTERING_INFORMATION` | Clustering verify cell |
| Result cache — zero-credit repeated queries | Business query cells |
| `QUERY_HISTORY` — execution metadata | Performance check cell |


---
## Cell 1 — Set context

Always set context explicitly at the top of every script. Never rely on whatever was set in a previous session.

**Why `USE SCHEMA SHOPFLOW_STAGED`?**

All source tables (`STG_*`) live in `SHOPFLOW_STAGED`. Setting it as the default schema means we can write `STG_ORDERS` instead of `SHOPFLOW_STAGED.STG_ORDERS` in every `FROM` clause. `CREATE TABLE` statements explicitly use `SHOPFLOW_ANALYTICS.*` to be unambiguous.


In [1]:
--USE WAREHOUSE SHOPFLOW_WH;
USE DATABASE  SHOPFLOW_DB;
USE SCHEMA    SHOPFLOW_STAGED;


SyntaxError: invalid syntax (3556290433.py, line 1)

In [ ]:
-- Confirm context before doing anything
SELECT
    CURRENT_WAREHOUSE() AS warehouse,
    CURRENT_DATABASE()  AS database,
    CURRENT_SCHEMA()    AS schema,
    CURRENT_ROLE()      AS role;


---
## Cell 2 — Confirm STAGED layer is ready

Before building ANALYTICS, verify the STAGED tables all exist and have data. If any table shows 0 rows, go back to Phase 2 and rebuild it before continuing.


In [ ]:
-- Confirm all 9 STAGED tables exist with row counts
-- Every table must have rows before we can build ANALYTICS from them
SELECT
    table_name,
    row_count,
    TO_CHAR(last_altered, 'YYYY-MM-DD HH24:MI') AS last_built
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_STAGED'
  AND table_name   LIKE 'STG_%'
ORDER BY table_name;


---
## Cells 3 – 7 — Build the star schema

### Read this before running any table cell

**Fact table vs dimension tables — the key distinction**

| | Fact table | Dimension tables |
|--|-----------|-----------------|
| What it stores | Events and measurements | Context about those events |
| Row count | Large — one row per event | Small — one row per entity |
| Column types | Mostly numeric measures + foreign keys | Mostly descriptive attributes |
| Changes over time | Append-only (new orders come in) | Slowly changing (seller moves city) |
| Example | FACT_ORDERS — price, freight, delay | DIM_SELLERS — city, state, lat/lng |

**Why item grain for FACT_ORDERS?**

We build FACT_ORDERS at the *order item* grain — one row per item, not one row per order. An order with 3 products from 3 sellers needs 3 rows because price, seller, and product are different for each item. Order grain would collapse them — losing the ability to `GROUP BY product` or `GROUP BY seller` cleanly.

**Foreign key convention**

Every row in FACT_ORDERS carries the keys needed to join to every dimension:
- `customer_id` → `DIM_CUSTOMERS`
- `product_id` → `DIM_PRODUCTS`
- `seller_id` → `DIM_SELLERS`
- `order_purchase_ts::DATE` → `DIM_DATE`

> **💡 Concept Note: Snowflake Foreign Keys**
>
> Snowflake supports `FOREIGN KEY` constraints in DDL but does not enforce them at write time — they are informational, marked `NOVALIDATE` and `RELY` by default. The query optimizer may use them for planning. Referential integrity is your responsibility to check (as you did in Phase 2 Cell 14).


---
## Cell 3 — Create FACT_ORDERS

### What this table contains
One row per order item. The centre of the star schema — every analytical question flows through this table.

### Columns and their purpose

**Foreign keys** (join to dimensions):
- `order_id`, `customer_id` → link to `DIM_CUSTOMERS` via `STG_ORDERS`
- `product_id` → `DIM_PRODUCTS`
- `seller_id` → `DIM_SELLERS`
- `order_purchase_ts::DATE` → `DIM_DATE`

**Measures** (numeric, aggregatable):
- `price` — item price in BRL
- `freight_value` — shipping cost in BRL
- `payment_value` — what the customer actually paid (from STG_PAYMENTS)
- `delivery_delay_days` — pre-computed in STAGED, carried forward

**Degenerate dimension** (kept in fact, not worth a separate table):
- `order_status` — low cardinality, no attributes, used in WHERE filters

### The JOIN pattern

`STG_ORDER_ITEMS` is the spine — one row per item. We JOIN `STG_ORDERS` to bring in timestamps and customer context, and LEFT JOIN `STG_PAYMENTS` aggregated by `order_id` to bring in payment value.

**Why LEFT JOIN for payments?**

Some orders in the Olist dataset have no payment record — cancelled or test orders. LEFT JOIN keeps all order items regardless. INNER JOIN would silently drop those rows.

> **💡 Concept Note: Aggregating before joining**
>
> `STG_PAYMENTS` has multiple rows per `order_id`. If you JOIN it directly, one order item would match multiple payment rows and you'd get row fan-out — inflated counts and wrong sums. The fix: aggregate payments to one row per `order_id` in a CTE first, then JOIN that aggregated result. This pattern is critical — fan-out from a bad JOIN is one of the most common sources of wrong analytics results.


In [ ]:
-- Pre-aggregate payments to one row per order BEFORE joining
-- Without this, each order item would fan-out to multiple payment rows
-- producing inflated row counts and wrong SUM(payment_value)
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.FACT_ORDERS AS
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value)          AS payment_value,  -- total paid for this order
        MAX(payment_type)           AS primary_payment_type,  -- most common method
        MAX(payment_installments)   AS max_installments
    FROM STG_PAYMENTS
    GROUP BY order_id
)
SELECT
    -- keys
    i.order_id,
    i.order_item_id,
    i.product_id,
    i.seller_id,
    o.customer_id,

    -- degenerate dimension
    o.order_status,

    -- timestamps
    o.order_purchase_ts,
    o.estimated_delivery_ts,
    o.delivered_ts,

    -- measures
    i.price,
    i.freight_value,
    i.price + i.freight_value           AS total_item_value,  -- convenience measure
    o.delivery_delay_days,
    p.payment_value,
    p.primary_payment_type,
    p.max_installments

FROM STG_ORDER_ITEMS i
JOIN STG_ORDERS o
  ON i.order_id = o.order_id
LEFT JOIN payment_totals p
  ON i.order_id = p.order_id
WHERE i.order_id    IS NOT NULL
  AND i.product_id  IS NOT NULL
  AND i.seller_id   IS NOT NULL;


In [ ]:
-- Verify FACT_ORDERS: row count, measure ranges, NULL check on keys
SELECT
    COUNT(*)                                AS total_rows,
    COUNT(DISTINCT order_id)                AS distinct_orders,
    COUNT(DISTINCT product_id)              AS distinct_products,
    COUNT(DISTINCT seller_id)               AS distinct_sellers,
    ROUND(MIN(price), 2)                    AS min_price_brl,
    ROUND(MAX(price), 2)                    AS max_price_brl,
    ROUND(AVG(price), 2)                    AS avg_price_brl,
    ROUND(SUM(total_item_value) / 1000000, 2) AS total_gmv_millions_brl,
    SUM(CASE WHEN delivery_delay_days > 0
             THEN 1 ELSE 0 END)             AS late_deliveries,
    SUM(CASE WHEN payment_value IS NULL
             THEN 1 ELSE 0 END)             AS orders_without_payment
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS;


---
## Cell 4 — Create DIM_CUSTOMERS

### What this table contains
One row per unique customer — using `customer_unique_id` as the key, not `customer_id`.

### The customer_id vs customer_unique_id distinction

This is the single most important thing to remember about the Olist dataset:

- `customer_id` — generated fresh for every order. One person who places 5 orders has 5 different `customer_id` values in `STG_CUSTOMERS`.
- `customer_unique_id` — the true person identifier. Use this as the dimension key.

`DIM_CUSTOMERS` uses `SELECT DISTINCT customer_unique_id` so each real person appears exactly once. Location comes from the most recent order's zip code via `STG_GEOLOCATION`.

### Why LEFT JOIN to geolocation?

Some customers have a zip code that doesn't exist in `STG_GEOLOCATION` — data quality gap in the source. LEFT JOIN keeps all customers. Customers without coordinates get NULL lat/lng — which is acceptable and visible, not silently dropped.


In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.DIM_CUSTOMERS AS
SELECT DISTINCT
    c.customer_unique_id    AS customer_id,  -- true unique person key
    c.customer_city         AS city,
    c.customer_state        AS state,
    g.lat,
    g.lng
FROM STG_CUSTOMERS c
LEFT JOIN STG_GEOLOCATION g
       ON c.customer_zip_code_prefix = g.zip_code_prefix
WHERE c.customer_unique_id IS NOT NULL;


In [ ]:
-- Verify DIM_CUSTOMERS: one row per unique person, state coverage
SELECT
    COUNT(*)                        AS total_customers,
    COUNT(DISTINCT state)           AS states_covered,
    SUM(CASE WHEN lat IS NULL
             THEN 1 ELSE 0 END)     AS customers_without_coordinates,
    ROUND(
        SUM(CASE WHEN lat IS NULL THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 1
    )                               AS pct_without_coordinates
FROM SHOPFLOW_ANALYTICS.DIM_CUSTOMERS;


---
## Cell 5 — Create DIM_PRODUCTS

### What this table contains
One row per product. The English category name is already available from Phase 2 — `STG_PRODUCTS` has `product_category_english` from the LEFT JOIN to the translation table done in STAGED.

### No join needed here

Unlike DIM_CUSTOMERS and DIM_SELLERS, DIM_PRODUCTS is a straight copy from a single STAGED table. All the hard work — fixing typos, casting dimensions, adding English names — was done in Phase 2. ANALYTICS just selects the clean columns it needs.

This is the Medallion Architecture paying off: by the time data reaches ANALYTICS, it is already clean, typed, and enriched. ANALYTICS cells are simpler and faster to write.

> **💡 Design Tip: What belongs in a dimension?**
>
> Include descriptive attributes analysts will filter or group by. Exclude raw metrics. For products: category, weight, dimensions — yes. For price: no — price changes per order and belongs in the fact table. A dimension should describe *what something is*, not *what happened to it*.


In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.DIM_PRODUCTS AS
SELECT
    product_id,
    product_category_name       AS category_portuguese,
    product_category_english    AS category_english,
    product_name_length,
    product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM STG_PRODUCTS
WHERE product_id IS NOT NULL;


In [ ]:
-- Verify DIM_PRODUCTS: row count, category coverage, uncategorised check
SELECT
    COUNT(*)                                    AS total_products,
    COUNT(DISTINCT category_english)            AS distinct_categories,
    SUM(CASE WHEN category_english = 'uncategorised'
             THEN 1 ELSE 0 END)                 AS uncategorised_products,
    SUM(CASE WHEN product_weight_g IS NULL
             THEN 1 ELSE 0 END)                 AS missing_weight
FROM SHOPFLOW_ANALYTICS.DIM_PRODUCTS;


---
## Cell 6 — Create DIM_SELLERS

### What this table contains
One row per seller. Same pattern as DIM_CUSTOMERS — LEFT JOIN to geolocation for coordinates.

### Why seller location matters

Sellers and customers are spread across 27 Brazilian states. Distance between seller and customer affects freight cost, delivery time, and ultimately review scores. Having lat/lng on both dimensions enables geographic analysis — for example: do sellers in SP deliver to SP customers faster than sellers in RS delivering to SP?

### Seller vs customer dimension — same pattern, different business meaning

Both use LEFT JOIN to geolocation on zip code prefix. Both keep NULL coordinates rather than dropping rows. The dimension key is `seller_id` — unlike customers, sellers don't have a `unique_id` vs `id` distinction. Each seller appears exactly once in `STG_SELLERS`.


In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.DIM_SELLERS AS
SELECT
    s.seller_id,
    s.seller_city   AS city,
    s.seller_state  AS state,
    g.lat,
    g.lng
FROM STG_SELLERS s
LEFT JOIN STG_GEOLOCATION g
       ON s.seller_zip_code_prefix = g.zip_code_prefix
WHERE s.seller_id IS NOT NULL;


In [ ]:
-- Verify DIM_SELLERS: row count, top states, coordinate coverage
SELECT
    COUNT(*)                        AS total_sellers,
    COUNT(DISTINCT state)           AS states_covered,
    SUM(CASE WHEN lat IS NULL
             THEN 1 ELSE 0 END)     AS sellers_without_coordinates
FROM SHOPFLOW_ANALYTICS.DIM_SELLERS;


---
## Cell 7 — Create DIM_DATE

### What this table contains
One row per calendar day from 2016-01-01 to 2018-12-31 — covering the full Olist dataset date range with buffer. No source table — this table is entirely generated.

### Why a date dimension?

Without DIM_DATE, every time query calculation calls a function on every row:
```sql
-- Without DIM_DATE — expensive, repeats function call per row
SELECT DATE_TRUNC('month', order_purchase_ts), SUM(price)
FROM FACT_ORDERS GROUP BY 1;

-- With DIM_DATE — filter on pre-computed attributes, no functions
SELECT d.year_month, SUM(f.price)
FROM FACT_ORDERS f
JOIN DIM_DATE d ON f.order_purchase_ts::DATE = d.date_key
GROUP BY d.year_month;
```

The real value is the pre-computed attributes: `year`, `month_num`, `month_name`, `quarter`, `day_of_week`, `is_weekend`. Analysts can filter and group by these without knowing any date functions.

### How row generation works in Snowflake

Snowflake has no `GENERATE_SERIES` like PostgreSQL. The standard pattern uses `TABLE(GENERATOR(ROWCOUNT => n))` which produces n rows, combined with `ROW_NUMBER()` to create an incrementing integer, then `DATEADD` to turn that integer into a date:

```sql
SELECT DATEADD('day', ROW_NUMBER() OVER (ORDER BY seq4()) - 1, '2016-01-01'::DATE)
FROM TABLE(GENERATOR(ROWCOUNT => 1096))
```

> **💡 SnowPro Tip: GENERATOR**
>
> `TABLE(GENERATOR(ROWCOUNT => n))` is Snowflake-specific syntax — no equivalent in standard SQL or Redshift. It generates a virtual table of n rows. Combined with `SEQ4()` (a built-in sequence generator) and window functions, it is the standard way to build a date dimension or any sequential range in Snowflake without a source table.


In [ ]:
-- Generate one row per calendar day: 2016-01-01 → 2018-12-31
-- 1096 days = 3 full years, covers entire Olist dataset with buffer
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.DIM_DATE AS
WITH date_spine AS (
    SELECT
        DATEADD('day',
            ROW_NUMBER() OVER (ORDER BY seq4()) - 1,
            '2016-01-01'::DATE
        ) AS date_key
    FROM TABLE(GENERATOR(ROWCOUNT => 1096))
)
SELECT
    date_key,
    YEAR(date_key)                              AS year,
    MONTH(date_key)                             AS month_num,
    MONTHNAME(date_key)                         AS month_name,
    QUARTER(date_key)                           AS quarter,
    DAYOFWEEK(date_key)                         AS day_of_week,  -- 0=Sun, 1=Mon ... 6=Sat
    DAYNAME(date_key)                           AS day_name,
    DAY(date_key)                               AS day_of_month,
    DAYOFYEAR(date_key)                         AS day_of_year,
    WEEKOFYEAR(date_key)                        AS week_of_year,
    TO_CHAR(date_key, 'YYYY-MM')                AS year_month,   -- '2017-03' for easy grouping
    CASE WHEN DAYOFWEEK(date_key) IN (0, 6)
         THEN TRUE ELSE FALSE END               AS is_weekend,
    CONCAT('Q', QUARTER(date_key), ' ',
           YEAR(date_key))                      AS quarter_label  -- 'Q1 2017'
FROM date_spine
ORDER BY date_key;


In [ ]:
-- Verify DIM_DATE: row count, date range, sample rows
SELECT
    COUNT(*)            AS total_days,
    MIN(date_key)       AS first_date,
    MAX(date_key)       AS last_date,
    COUNT(DISTINCT year) AS years_covered
FROM SHOPFLOW_ANALYTICS.DIM_DATE;


In [ ]:
-- Sample a few rows to confirm attributes look correct
SELECT *
FROM SHOPFLOW_ANALYTICS.DIM_DATE
WHERE date_key IN ('2016-01-01', '2017-03-15', '2018-11-23', '2018-11-24')
ORDER BY date_key;


---
## Cell 8 — Verify all ANALYTICS tables

Query `INFORMATION_SCHEMA.TABLES` to confirm all 5 tables exist with expected row counts.

### Expected counts
- `FACT_ORDERS` — ~112,000 rows (one per order item)
- `DIM_CUSTOMERS` — ~96,000 rows (unique people, fewer than customer_id count)
- `DIM_PRODUCTS` — ~32,951 rows
- `DIM_SELLERS` — ~3,095 rows
- `DIM_DATE` — exactly 1,096 rows


In [ ]:
SELECT
    table_name,
    row_count,
    TO_CHAR(created, 'YYYY-MM-DD HH24:MI') AS created_at
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_ANALYTICS'
ORDER BY table_name;


---
## Cell 9 — Referential integrity checks

Every `product_id` in `FACT_ORDERS` must exist in `DIM_PRODUCTS`. Every `seller_id` must exist in `DIM_SELLERS`. Orphan keys mean JOINs will produce NULLs in dimension columns — silently wrong results in every downstream report.

Run each check. All should return 0.


In [ ]:
-- Check 1: FACT_ORDERS rows with no matching product in DIM_PRODUCTS
SELECT COUNT(*) AS orphan_products
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
LEFT JOIN SHOPFLOW_ANALYTICS.DIM_PRODUCTS p
       ON f.product_id = p.product_id
WHERE p.product_id IS NULL;


In [ ]:
-- Check 2: FACT_ORDERS rows with no matching seller in DIM_SELLERS
SELECT COUNT(*) AS orphan_sellers
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
LEFT JOIN SHOPFLOW_ANALYTICS.DIM_SELLERS s
       ON f.seller_id = s.seller_id
WHERE s.seller_id IS NULL;


In [ ]:
-- Check 3: FACT_ORDERS dates outside DIM_DATE range
-- Any order purchase date not covered by DIM_DATE will fail to join
SELECT COUNT(*) AS dates_outside_dim_range
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
LEFT JOIN SHOPFLOW_ANALYTICS.DIM_DATE d
       ON f.order_purchase_ts::DATE = d.date_key
WHERE f.order_purchase_ts IS NOT NULL
  AND d.date_key IS NULL;


---
## Cells 10 – 13 — The 4 core business queries

These are the questions the entire project was built to answer. Every cell from Phase 0 to here was working toward being able to run these cleanly.

Run each one. Note the execution time — you will compare it after adding a clustering key in Cell 14.

> **💡 Concept Note: Result Cache**
>
> Snowflake has three cache layers:
> - **Metadata cache** — `SHOW` commands, `INFORMATION_SCHEMA` queries — free, no warehouse needed
> - **Result cache** — repeated identical queries return the same result instantly, 0 credits, 24-hour TTL
> - **Local disk cache** — warehouse caches recently scanned micro-partitions in memory
>
> After running each business query once, run it a second time immediately. You will see `0ms` execution time and a note in `QUERY_HISTORY` that the result came from cache. This is a SnowPro exam topic — result cache is account-level, shared across all users, and invalidated when the underlying table changes.


---
## Cell 10 — Business Query 1: GMV by product category

**Question:** Which product categories generate the most revenue?

This is a direct join from FACT_ORDERS to DIM_PRODUCTS, grouped by category. The answer tells the business where to focus seller acquisition and marketing spend.


In [ ]:
-- Business Query 1: GMV by product category
-- Join FACT to DIM_PRODUCTS for category name
SELECT
    p.category_english,
    COUNT(DISTINCT f.order_id)                      AS orders,
    COUNT(*)                                        AS items_sold,
    ROUND(SUM(f.price), 2)                          AS revenue_brl,
    ROUND(SUM(f.freight_value), 2)                  AS freight_brl,
    ROUND(SUM(f.total_item_value), 2)               AS gmv_brl,
    ROUND(AVG(f.price), 2)                          AS avg_item_price_brl,
    ROUND(SUM(f.freight_value) / NULLIF(SUM(f.price), 0) * 100, 1) AS freight_pct_of_revenue
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
JOIN SHOPFLOW_ANALYTICS.DIM_PRODUCTS p
  ON f.product_id = p.product_id
WHERE f.order_status = 'delivered'
GROUP BY p.category_english
ORDER BY gmv_brl DESC
LIMIT 15;


---
## Cell 11 — Business Query 2: Seller cancellation rates

**Question:** Which sellers have the highest cancellation rates?

A high cancellation rate signals a seller reliability problem. The business can use this to enforce quality standards, remove bad sellers, or trigger seller support interventions.

Note the use of `NULLIF` in the cancellation rate calculation — dividing by zero would crash the query if a seller has no delivered orders.


In [ ]:
-- Business Query 2: Seller cancellation rates
-- Only include sellers with enough volume to be statistically meaningful
SELECT
    f.seller_id,
    s.state                                         AS seller_state,
    COUNT(DISTINCT f.order_id)                      AS total_orders,
    COUNT(DISTINCT CASE WHEN f.order_status = 'canceled'
                        THEN f.order_id END)         AS cancelled_orders,
    ROUND(
        COUNT(DISTINCT CASE WHEN f.order_status = 'canceled'
                            THEN f.order_id END) * 100.0
        / NULLIF(COUNT(DISTINCT f.order_id), 0),
    1)                                              AS cancellation_rate_pct,
    ROUND(SUM(CASE WHEN f.order_status = 'delivered'
                   THEN f.price ELSE 0 END), 2)     AS delivered_revenue_brl
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
JOIN SHOPFLOW_ANALYTICS.DIM_SELLERS s
  ON f.seller_id = s.seller_id
GROUP BY f.seller_id, s.state
HAVING COUNT(DISTINCT f.order_id) >= 20  -- minimum volume threshold
ORDER BY cancellation_rate_pct DESC
LIMIT 20;


---
## Cell 12 — Business Query 3: Delivery delay vs review score

**Question:** How does delivery delay affect review scores?

This is the most analytically interesting query in the project — it joins FACT_ORDERS to STG_REVIEWS (reviews aren't a dimension, they're a separate fact) and buckets orders by delivery performance.

The result should show a clear pattern: early deliveries correlate with high scores, late deliveries with low scores. If it doesn't, that itself is an interesting finding worth investigating.


In [ ]:
-- Business Query 3: Delivery delay vs review score
-- JOIN FACT_ORDERS back to STAGED reviews (reviews are not in the star schema)
SELECT
    CASE
        WHEN f.delivery_delay_days <= -7  THEN '> 1 week early'
        WHEN f.delivery_delay_days <   0  THEN 'Early'
        WHEN f.delivery_delay_days =   0  THEN 'On time'
        WHEN f.delivery_delay_days <=  7  THEN 'Up to 1 week late'
        WHEN f.delivery_delay_days <= 14  THEN '1–2 weeks late'
        ELSE                                   'More than 2 weeks late'
    END                                         AS delivery_bucket,
    COUNT(DISTINCT f.order_id)                  AS orders,
    ROUND(AVG(r.review_score), 2)               AS avg_review_score,
    COUNT(DISTINCT CASE WHEN r.review_score >= 4
                        THEN f.order_id END)    AS happy_customers,
    COUNT(DISTINCT CASE WHEN r.review_score <= 2
                        THEN f.order_id END)    AS unhappy_customers
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
JOIN SHOPFLOW_STAGED.STG_REVIEWS r
  ON f.order_id = r.order_id
WHERE f.delivery_delay_days IS NOT NULL
GROUP BY delivery_bucket
ORDER BY AVG(f.delivery_delay_days);


---
## Cell 13 — Business Query 4: Monthly GMV trend

**Question:** What is the monthly GMV trend?

This query uses DIM_DATE to group by `year_month` — no date function needed on the fact table column. The pre-computed `year_month` attribute does the work.

The Olist dataset runs from late 2016 to mid-2018. You should see steady growth through 2017 with a spike around November 2017 (Black Friday in Brazil) and a partial 2018.


In [ ]:
-- Business Query 4: Monthly GMV trend
-- Use DIM_DATE year_month attribute — no DATE_TRUNC needed on fact table
SELECT
    d.year_month,
    d.year,
    d.month_name,
    COUNT(DISTINCT f.order_id)                  AS orders,
    COUNT(*)                                    AS items_sold,
    ROUND(SUM(f.total_item_value), 2)           AS gmv_brl,
    ROUND(AVG(f.price), 2)                      AS avg_item_price,
    ROUND(
        (SUM(f.total_item_value) - LAG(SUM(f.total_item_value))
            OVER (ORDER BY d.year_month))
        / NULLIF(LAG(SUM(f.total_item_value))
            OVER (ORDER BY d.year_month), 0) * 100
    , 1)                                        AS mom_growth_pct
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
JOIN SHOPFLOW_ANALYTICS.DIM_DATE d
  ON f.order_purchase_ts::DATE = d.date_key
WHERE f.order_status = 'delivered'
GROUP BY d.year_month, d.year, d.month_name
ORDER BY d.year_month;


---
## Cell 14 — Add a clustering key to FACT_ORDERS

### What is a clustering key?

Snowflake stores table data in **micro-partitions** — immutable, compressed files of roughly 16MB each. Snowflake maintains metadata about the min and max value of every column in every micro-partition. When you filter on a column, Snowflake uses this metadata to skip partitions that can't contain matching rows — called **partition pruning**.

By default, micro-partitions are organised in insertion order. If you always filter `FACT_ORDERS` by `order_purchase_ts` (e.g. "last 30 days"), Snowflake has to scan every partition because date values are scattered randomly.

A **clustering key** tells Snowflake to reorganise the table so rows with similar values of the cluster key are co-located in the same partitions. After clustering on `order_purchase_ts`, a "last 30 days" filter prunes most partitions — scanning far less data.

### Before and after

Run Business Query 4 (Cell 13) now and note the execution time from `QUERY_HISTORY`. Then add the clustering key and run the same query again. Compare.

> **💡 SnowPro Tip: Clustering**
>
> - Cluster on columns most commonly used in `WHERE` and `JOIN` filters
> - Multi-column clustering: `CLUSTER BY (col1, col2)` — col1 is primary, col2 breaks ties
> - `SYSTEM$CLUSTERING_INFORMATION('table')` — shows cluster depth and health score
> - Automatic clustering = Snowflake maintains the clustering for you (extra cost, hands-free)
> - Clustering is most impactful on large tables (hundreds of millions of rows) — on this 112k-row dataset the improvement will be visible but modest


In [ ]:
-- Step 1: Check execution time of the monthly GMV query BEFORE clustering
-- Note the 'partitions_scanned' and 'partitions_total' in QUERY_HISTORY
SELECT
    query_text,
    TO_CHAR(start_time, 'HH24:MI:SS')  AS run_at,
    execution_time / 1000               AS execution_seconds,
    partitions_scanned,
    partitions_total,
    ROUND(partitions_scanned * 100.0
          / NULLIF(partitions_total, 0), 1) AS pct_partitions_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 5))
WHERE query_text ILIKE '%year_month%'
   OR query_text ILIKE '%gmv_brl%'
ORDER BY start_time DESC
LIMIT 3;


In [ ]:
-- Add clustering key on order_purchase_ts and seller_id
-- order_purchase_ts — most common WHERE filter (date range queries)
-- seller_id — secondary, helps seller-specific dashboard queries
ALTER TABLE SHOPFLOW_ANALYTICS.FACT_ORDERS
    CLUSTER BY (DATE_TRUNC('month', order_purchase_ts), seller_id);


In [ ]:
-- Check clustering health — average_depth close to 1.0 means well-clustered
SELECT SYSTEM$CLUSTERING_INFORMATION(
    'SHOPFLOW_ANALYTICS.FACT_ORDERS',
    '(DATE_TRUNC(''month'', order_purchase_ts), seller_id)'
);


In [ ]:
-- Re-run Business Query 4 after clustering — compare execution time and partitions scanned
-- Run it TWICE: first run populates the cache, second run returns from result cache (0ms)
SELECT
    d.year_month,
    d.year,
    d.month_name,
    COUNT(DISTINCT f.order_id)                  AS orders,
    COUNT(*)                                    AS items_sold,
    ROUND(SUM(f.total_item_value), 2)           AS gmv_brl,
    ROUND(AVG(f.price), 2)                      AS avg_item_price,
    ROUND(
        (SUM(f.total_item_value) - LAG(SUM(f.total_item_value))
            OVER (ORDER BY d.year_month))
        / NULLIF(LAG(SUM(f.total_item_value))
            OVER (ORDER BY d.year_month), 0) * 100
    , 1)                                        AS mom_growth_pct
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
JOIN SHOPFLOW_ANALYTICS.DIM_DATE d
  ON f.order_purchase_ts::DATE = d.date_key
WHERE f.order_status = 'delivered'
GROUP BY d.year_month, d.year, d.month_name
ORDER BY d.year_month;


---
## Cell 15 — Suspend warehouse

Always suspend the warehouse explicitly at the end of a session. `SHOPFLOW_WH` has `AUTO_SUSPEND=60` but suspending manually stops the billing clock immediately.


In [ ]:
ALTER WAREHOUSE SHOPFLOW_WH SUSPEND;


In [ ]:
SHOW WAREHOUSES LIKE 'SHOPFLOW%';


---
## 🛠 Self-Guided Exploration:

Now that your star schema is built and the business queries are running, test your understanding of data modelling and query patterns. Write these from scratch.

### Challenge 1 (Easy): Explore the date dimension
**Objective:** Practice querying a date dimension for time-based filtering.

**Task:** Write a query against `DIM_DATE` to return every row where `quarter_label` is `'Q4 2017'`, showing `date_key`, `day_name`, and `is_weekend`. Order by `date_key`.

> 💡 **Think about it:** How many rows do you expect? How many of them should have `is_weekend = TRUE`?


In [ ]:
-- Write your SQL for Challenge 1 here


### Challenge 2 (Medium): Fan-out trap
**Objective:** Understand why pre-aggregating before a JOIN matters.

**Task:** Write a query that JOINs `FACT_ORDERS` directly to `STG_PAYMENTS` (without pre-aggregating) on `order_id`, and counts the total rows returned. Then compare that number to `COUNT(*)` on `FACT_ORDERS` alone.

> 💡 **Think about it:** Why are the two counts different? What does the difference tell you about the data? What would happen to `SUM(price)` if you used this direct JOIN to calculate GMV?


In [ ]:
-- Write your SQL for Challenge 2 here


### Challenge 3 (Hard): Top sellers by state
**Objective:** Multi-table star schema join with aggregation and filtering.

**Task:** Write a query that finds the top 3 sellers by total `revenue_brl` within each Brazilian state. Only include delivered orders. Show `state`, `seller_id`, `revenue_brl`, and `rank_in_state`.

> 💡 **Hint:** You will need `FACT_ORDERS`, `DIM_SELLERS`, and a window function to rank within each state. Think about which window function gives you a clean 1-2-3 rank with no gaps.


In [ ]:
-- Write your SQL for Challenge 3 here


### Challenge 4 (Complex): Weekend vs weekday order patterns
**Objective:** Use the date dimension to answer a question that would require date functions without it.

**Task:** Compare the average order value (`AVG(total_item_value)`) and order count between weekend and weekday orders for delivered orders only. Show whether weekends or weekdays have higher average order value.

> 💡 **Think about it:** Could you answer this question without `DIM_DATE`? What function would you need on `order_purchase_ts` to determine weekend vs weekday — and why is `DIM_DATE.is_weekend` cleaner?


In [ ]:
-- Write your SQL for Challenge 4 here


---
## Phase 3 complete

If Cell 8 shows all 5 ANALYTICS tables, Cell 9 shows 0 orphan records, and Cells 10–13 return meaningful business results — your star schema is fully operational.

### What you just built

```text
SHOPFLOW_ANALYTICS
├── FACT_ORDERS         ← ~112k rows · price · freight · delay · payment
├── DIM_CUSTOMERS       ← ~96k unique people · city · state · lat/lng
├── DIM_PRODUCTS        ← ~33k products · English category · dimensions
├── DIM_SELLERS         ← ~3k sellers · city · state · lat/lng
└── DIM_DATE            ← 1,096 days · year/month/quarter/weekend flag
```

### The 4 business questions — answered

| Question | Query | Key join |
|----------|-------|----------|
| Which categories generate most revenue? | Cell 10 | FACT → DIM_PRODUCTS |
| Which sellers have highest cancellation rates? | Cell 11 | FACT → DIM_SELLERS |
| How does delivery delay affect reviews? | Cell 12 | FACT → STG_REVIEWS |
| What is the monthly GMV trend? | Cell 13 | FACT → DIM_DATE |

### Key Snowflake concepts you practised

| Topic | Covered |
|-------|---------|
| Star schema — fact vs dimension design | ✅ |
| CTE to pre-aggregate before JOIN (fan-out prevention) | ✅ |
| DISTINCT in dimension builds | ✅ |
| GENERATOR + ROW_NUMBER for date spine | ✅ |
| NULLIF for safe division | ✅ |
| LAG window function for month-over-month growth | ✅ |
| Clustering keys — CLUSTER BY | ✅ |
| SYSTEM$CLUSTERING_INFORMATION | ✅ |
| Result cache — 0-credit repeated queries | ✅ |
| QUERY_HISTORY — partitions scanned | ✅ |

### Next — Phase 4: Performance + Security

- RBAC: `SHOPFLOW_ENGINEER`, `SHOPFLOW_ANALYST`, `SHOPFLOW_SELLER` roles
- Dynamic data masking on customer PII
- Row access policy — sellers see only their own rows
- Result cache deep-dive — demonstrate zero-credit repeated queries
